In [0]:
dbutils.widgets.text("catalogo", "mlops", "Catálogo")
dbutils.widgets.text("esquema", "frecuencia", "Esquema")
dbutils.widgets.text("archivo", "freMTPL2freq.csv", "Archivo")
dbutils.widgets.text("tope_claimnb", "4", "Tope ClaimNb")
dbutils.widgets.text("tope_exposure", "1", "Tope Exposure")

catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
archivo = dbutils.widgets.get("archivo")

ruta = f"/Volumes/{catalogo}/{esquema}/raw/{archivo}"

In [0]:
df_texto = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(ruta)
)

df_texto.printSchema()
display(df_texto.limit(10))
print(df_texto.count())

In [0]:
from pyspark.sql import functions as F

tipos = {
    "IDpol": "bigint",
    "ClaimNb": "int",
    "Exposure": "double",
    "Area": "string",
    "VehPower": "int",
    "VehAge": "int",
    "DrivAge": "int",
    "BonusMalus": "int",
    "VehBrand": "string",
    "VehGas": "string",
    "Density": "int",
    "Region": "string",
}

df = df_texto.select([
    F.expr(f"try_cast(try_cast(`{c}` AS double) AS bigint)").alias(c) if c == "IDpol"
    else F.expr(f"try_cast(`{c}` AS {t})").alias(c)
    for c, t in tipos.items()
])

def contar_nulos(d, etapa):
    return d.select(
        F.lit(etapa).alias("etapa"),
        *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in tipos],
    )

df.printSchema()
display(contar_nulos(df_texto, "texto").unionByName(contar_nulos(df, "tipado")))

In [0]:
tabla_raw = f"{catalogo}.{esquema}.polizas_raw"

df.write.mode("overwrite").saveAsTable(tabla_raw)

print(spark.table(tabla_raw).count())
display(spark.sql(f"DESCRIBE HISTORY {tabla_raw}"))

In [0]:
pol = spark.table(tabla_raw)
total = pol.count()

display(
    pol.groupBy("ClaimNb")
    .agg(F.count("*").alias("polizas"), F.sum("Exposure").alias("exposicion"))
    .withColumn("pct_polizas", F.round(100 * F.col("polizas") / total, 3))
    .orderBy("ClaimNb")
)

display(pol.select(
    F.min("Exposure").alias("minimo"),
    F.expr("percentile(Exposure, array(0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99))").alias("p1_p5_p25_p50_p75_p95_p99"),
    F.max("Exposure").alias("maximo"),
    F.sum((F.col("Exposure") > 1).cast("int")).alias("mayores_a_1"),
))

dup = pol.groupBy("IDpol").count().filter("count > 1")
print(dup.count())

In [0]:
dbutils.widgets.text("tope_claimnb", "4", "Tope ClaimNb")
dbutils.widgets.text("tope_exposure", "1", "Tope Exposure")

In [0]:
tope_claimnb = int(dbutils.widgets.get("tope_claimnb"))
tope_exposure = float(dbutils.widgets.get("tope_exposure"))
pct_train, pct_test = 60, 20

limpio = (
    spark.table(tabla_raw)
    .withColumn("ClaimNb", F.least(F.col("ClaimNb"), F.lit(tope_claimnb)))
    .withColumn("Exposure", F.least(F.col("Exposure"), F.lit(tope_exposure)))
    .withColumn("bucket", F.pmod(F.xxhash64("IDpol"), F.lit(100)))
)

particiones = {
    "polizas_train": F.col("bucket") < pct_train,
    "polizas_test": (F.col("bucket") >= pct_train) & (F.col("bucket") < pct_train + pct_test),
    "polizas_nuevas": F.col("bucket") >= pct_train + pct_test,
}

resumen = None
for nombre, condicion in particiones.items():
    tabla = f"{catalogo}.{esquema}.{nombre}"
    limpio.filter(condicion).drop("bucket").write.mode("overwrite").saveAsTable(tabla)
    r = (
        spark.table(tabla)
        .agg(
            F.count("*").alias("polizas"),
            F.sum("Exposure").alias("exposicion"),
            (F.sum("ClaimNb") / F.sum("Exposure")).alias("frecuencia"),
        )
        .withColumn("tabla", F.lit(nombre))
    )
    resumen = r if resumen is None else resumen.unionByName(r)

display(resumen)